In [1]:
import mysql.connector
from mysql.connector import Error
import pandas as pd

In [4]:
try:
    conn = mysql.connector.connect(
        port="3307",
        user="root",
        password="",
        database="uber_eats_bangalore"
    )
    cursor=conn.cursor()
    print("MySQL Connected Successfully!")
except Error as e:
    print("Connection Error:", e)

MySQL Connected Successfully!


In [5]:
cursor.execute("CREATE DATABASE IF NOT EXISTS uber_eats_bangalore")
cursor.execute("USE uber_eats_bangalore")

cursor.execute('''
CREATE TABLE IF NOT EXISTS restaurants (
    restaurant_id INT AUTO_INCREMENT PRIMARY KEY,
    restaurant_name VARCHAR(255),
    location VARCHAR(100),
    cuisines TEXT,
    rate FLOAT,
    approx_cost_for_two FLOAT,
    online_order TINYINT,
    book_table TINYINT,
    price_segment VARCHAR(50)
);
''')
print("Table 'restaurants' is ready!")

Table 'restaurants' is ready!


In [6]:
df = pd.read_csv('data/cleaned/restaurants_cleaned.csv')

for i, row in df.iterrows():
    cursor.execute("""
        INSERT INTO restaurants 
        (restaurant_name, location, cuisines, rate, approx_cost_for_two, 
         online_order, book_table, price_segment)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        row['name'], row['location'], str(row.get('cuisines', '')),
        float(row['rate']), float(row['approx_cost(for two people)']),
        int(row['online_order']), int(row['book_table']), str(row['price_segment'])
    ))

conn.commit()
print(f"Successfully loaded {len(df)} restaurants into MySQL!")

Successfully loaded 23158 restaurants into MySQL!


In [7]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id VARCHAR(100) PRIMARY KEY,
    restaurant_name VARCHAR(255),
    order_date DATE,
    order_value FLOAT,
    discount_used VARCHAR(10),
    payment_method VARCHAR(20)
);
""")

In [8]:
orders_df = pd.read_csv('data/cleaned/orders_cleaned.csv')
for i, row in orders_df.iterrows():
    cursor.execute("""
        INSERT INTO orders VALUES (%s, %s, %s, %s, %s, %s)
    """, (
        row['order_id'], row['restaurant_name'], row['order_date'], 
        row['order_value'], row['discount_used'], row['payment_method']
    ))

conn.commit()
print(f"Successfully loaded {len(df)} oders into MySQL!")

Successfully loaded 23158 oders into MySQL!
